# VisionShield - train PPE model v2 (YOLO11)
Runtime -> Change runtime type -> **GPU**, then run the cells in order.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# Private repo: add a GitHub token as Colab secret GH_TOKEN (key icon in the left sidebar)
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git clone https://{token}@github.com/phokin-lang/VisionShield.git
%cd /content/VisionShield/ml

In [ ]:
# Keep training runs on Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/VisionShield/runs
!rm -rf runs && ln -s /content/drive/MyDrive/VisionShield/runs runs

In [ ]:
# Load the Roboflow 6-class PPE export (YOLOv8 format) into ml/dataset/
# Option A: zip on Drive
DATASET_ZIP = '/content/drive/MyDrive/VisionShield/ppe_dataset.zip'
!rm -rf dataset && mkdir -p dataset && unzip -q "$DATASET_ZIP" -d dataset
# Option B: Roboflow API
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# rf.workspace('WORKSPACE').project('PROJECT').version(1).download('yolov8', location='dataset')
!ls dataset

In [ ]:
!python prepare_dataset.py --src dataset --dst dataset_ppe --keep Hard_hat Vest

In [ ]:
!python train_model.py --model yolo11n.pt --data dataset_ppe/data.yaml

In [ ]:
# Run this instead of the cell above if the session disconnected mid-training
!python train_model.py --resume --data dataset_ppe/data.yaml

In [ ]:
!python compare_models.py --old model/ppe_hat_vest_nano.pt --new runs/detect/ppe_v2/weights/best.pt --split test

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob
res = YOLO('runs/detect/ppe_v2/weights/best.pt').predict('../backend/assets/cameras', conf=0.35, save=True, project='runs/predict', name='cameras', exist_ok=True)
for f in sorted(glob.glob('runs/predict/cameras/*'))[:8]:
    display(Image(filename=f, width=640))

In [ ]:
!pip install -q onnx onnxslim
!python export_onnx.py --ppe runs/detect/ppe_v2/weights/best.pt --only ppe
from google.colab import files
files.download('../backend/models/ppe_hat_vest_nano.onnx')
files.download('runs/detect/ppe_v2/weights/best.pt')